<strong><b><font size="5">Data Science Academy</font></b></strong>

<strong><b><font size="5">Web Scraping e Análise de Dados</font></b></strong>

<strong><b><font size="5">Lab 2</font></b></strong>

<strong><b><font size="4">Web Scraping e Análise Estatística - Avaliando o Desempenho dos Participantes em Corridas de 10 KM com Dados de Sensores RFID</font></b></strong>

![title](imagens/lab2.png)

## Definição do Problema

Você gosta de correr? Corrida é um dos esportes mais praticados em todo mundo, além de uma atividade saudável e prazerosa.

Neste Lab vamos coletar dados reais de participantes de uma corrida de 10 KM realizada em Hillsboro no Oregon, EUA, em junho de 2017. 

Resumo da corrida:

- Pessoas que completaram a corrida: 577
- Homens: 414
- Mulheres: 163

O objetivo desta análise é encontrar:
    
- O tempo médio de conclusão da corrida.
- Se os tempos de chegada dos corredores seguem uma distribuição normal.
- Analisar se há diferença de desempenho entre homens e mulheres de várias faixas etárias.

Os tempos registrados na corrida (bem como outros dados) são bem precisos pois os participantes usaram sensores RFID de onde os dados foram extraídos e disponibilizados na web. Mais detalhes sobre o uso dos sensores RFID aqui:

https://www.hubertiming.com/runners/faq

## Fonte de Dados

Como fonte de dados usaremos o web site <a href="https://www.hubertiming.com/">Huber Timing</a> que contém o resultado das corridas que acontecem nos EUA, bem como a agenda de novos eventos.

Aqui a página com o resultado da corrida que faremos a análise:

https://www.hubertiming.com/results/2017GPTR10K

## Carregando os Pacotes Usados Neste Jupyter Notebook

In [63]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python Usada Neste Jupyter Notebook:', python_version())

Versão da Linguagem Python Usada Neste Jupyter Notebook: 3.12.4


In [64]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook, se necessário.

In [65]:
# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

In [66]:
# Usaremos o pacote Pingouin para algumas das análises estatísticas
# https://pingouin-stats.org/index.html
!pip install -q pingouin

In [67]:
# Imports
import re
import requests
import matplotlib as m
import numpy as np
import pandas as pd
import pingouin as pg
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup as bs
from urllib.request import urlopen
from scipy import stats

# Imports para formatação dos gráficos
sns.set_style('whitegrid')
%matplotlib inline

In [68]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Data Science Academy" --iversions

Author: Data Science Academy

json      : 2.0.9
requests  : 2.32.2
numpy     : 1.26.4
matplotlib: 3.8.4
pandas    : 2.2.2
seaborn   : 0.13.2
scipy     : 1.13.1
pingouin  : 0.5.4
re        : 2.2.1



## Web Scraping

In [70]:
# Definindo a URL com o resultado da corrida
url = 'https://www.hubertiming.com/results/2017GPTR'

In [71]:
# Abre a conexão com a url
html = urlopen(url)

In [72]:
# Extrai o código HTML
soup = bs(html,'lxml')

In [73]:
# Extrai o texto do código html
texto = soup.get_text()

In [74]:
# Vamos visualizar as 10 primeiras linhas
linhas = soup.find_all('tr')
print(linhas[:10])

[<tr colspan="2">
<b>5K:</b>
</tr>, <tr>
<td>Finishers:</td>
<td>1458</td>
</tr>, <tr>
<td>Male:</td>
<td>771</td>
</tr>, <tr>
<td>Female:</td>
<td>687</td>
</tr>, <tr class="header">
<th>Place</th>
<th>Bib</th>
<th>Name</th>
<th>Gender</th>
<th>City</th>
<th>State</th>
<th>Time</th>
<th>Gun Time</th>
</tr>, <tr data-bib="2320">
<td>1</td>
<td>2320</td>
<td>

                    DANIEL M HINCKLEY

                </td>
<td>M</td>
<td>HILLSBORO</td>
<td>OR</td>
<td>16:42</td>
<td>16:44</td>
</tr>, <tr data-bib="2335">
<td>2</td>
<td>2335</td>
<td>

                    KORY F GRAY

                </td>
<td>M</td>
<td>HILLSBORO</td>
<td>OR</td>
<td>17:34</td>
<td>17:35</td>
</tr>, <tr data-bib="1770">
<td>3</td>
<td>1770</td>
<td>

                    FILIP SCHMOLE

                </td>
<td>M</td>
<td>PORTLAND</td>
<td>OR</td>
<td>18:13</td>
<td>18:14</td>
</tr>, <tr data-bib="2584">
<td>4</td>
<td>2584</td>
<td>

                    TRENTON C ROLLING

                </td>
<td>M</td>
<

## Limpeza e Transformação

Agora começa o trabalho pesado. Não fique apenas executando código e compreenda o que está sendo feito. Se tiver dificuldades, pergunte! Temos vários canais de suporte a sua disposição.

Começaremos obtendo todas as linhas da tabela em forma de lista e depois converteremos essa lista em um dataframe para facilitar a manipulação.

In [77]:
# Loop
for linha in linhas:
    linha_td = linha.find_all('td')

In [78]:
print(linha_td)

[<td>1458</td>, <td>1400</td>, <td>

                    SUMALATHA PURMA

                </td>, <td>F</td>, <td>PORTLAND</td>, <td>OR</td>, <td>1:48:13</td>, <td>1:48:13</td>]


Com expressões regulares em Python, vamos extrair os dados das tags HTML.

In [80]:
# Lista vazia para receber as linhas
lista_linhas = []

In [81]:
# Loop
for linha in linhas:
    cells = linha.find_all('td')
    str_cells = str(cells)
    clean = re.compile('<.*?>')
    clean2 = (re.sub(clean, '', str_cells))
    lista_linhas.append(clean2)

In [82]:
print(clean2)

[1458, 1400, 

                    SUMALATHA PURMA

                , F, PORTLAND, OR, 1:48:13, 1:48:13]


Convertemos a lista em um dataframe e obtemos uma visão rápida das 10 primeiras linhas usando o Pandas.

In [84]:
# Convertemos a lista em dataframe
df = pd.DataFrame(lista_linhas)

In [85]:
df.head(10)

,0
0,[]
1,"[Finishers:, 1458]"
2,"[Male:, 771]"
3,"[Female:, 687]"
4,[]
5,"[1, 2320, \r\n\r\n DANIEL M..."
6,"[2, 2335, \r\n\r\n KORY F G..."
7,"[3, 1770, \r\n\r\n FILIP SC..."
8,"[4, 2584, \r\n\r\n TRENTON ..."
9,"[5, 2688, \r\n\r\n YEAN-AN ..."


Claramente precisamos limpar os dados. Vamos ao trabalho.

In [87]:
# Dividindo cada linha e separando os dados a cada vírgula
df1 = df[0].str.split(',', expand = True)

In [88]:
df1.head(10)

,0,1,2,3,4,5,6,7
0,[],None,None,None,None,None,None,None
1,[Finishers:,1458],None,None,None,None,None,None
2,[Male:,771],None,None,None,None,None,None
3,[Female:,687],None,None,None,None,None,None
4,[],None,None,None,None,None,None,None
5,[1,2320,\r\n\r\n DANIEL M HINCKLEY...,M,HILLSBORO,OR,16:42,16:44]
6,[2,2335,\r\n\r\n KORY F GRAY\r\n\r...,M,HILLSBORO,OR,17:34,17:35]
7,[3,1770,\r\n\r\n FILIP SCHMOLE\r\n...,M,PORTLAND,OR,18:13,18:14]
8,[4,2584,\r\n\r\n TRENTON C ROLLING...,M,PORTLAND,OR,18:32,18:35]
9,[5,2688,\r\n\r\n YEAN-AN LIAO\r\n\...,M,HILLSBORO,OR,19:12,19:18]


Está começando a ficar bonito. Vamos fazer mais algumas limpezas. Leia e compreenda o código. Se tiver dúvidas, pergunte (embora isso não deva ser novidade se você fez o curso gratuito de <a href="https://www.datascienceacademy.com.br/course?courseid=python-fundamentos">Python Fundamentos</a> aqui na DSA).

In [90]:
# Removemos colchetes de abertura (lado esquerdo) do texto
df1[0] = df1[0].str.strip('[')

In [91]:
df1.head(10)

,0,1,2,3,4,5,6,7
0,],None,None,None,None,None,None,None
1,Finishers:,1458],None,None,None,None,None,None
2,Male:,771],None,None,None,None,None,None
3,Female:,687],None,None,None,None,None,None
4,],None,None,None,None,None,None,None
5,1,2320,\r\n\r\n DANIEL M HINCKLEY...,M,HILLSBORO,OR,16:42,16:44]
6,2,2335,\r\n\r\n KORY F GRAY\r\n\r...,M,HILLSBORO,OR,17:34,17:35]
7,3,1770,\r\n\r\n FILIP SCHMOLE\r\n...,M,PORTLAND,OR,18:13,18:14]
8,4,2584,\r\n\r\n TRENTON C ROLLING...,M,PORTLAND,OR,18:32,18:35]
9,5,2688,\r\n\r\n YEAN-AN LIAO\r\n\...,M,HILLSBORO,OR,19:12,19:18]


In [92]:
# Extraímos o cabeçalho de todas as tabelas no código HTML
col_labels = soup.find_all('th')

In [93]:
col_labels

[<th>Place</th>,
 <th>Bib</th>,
 <th>Name</th>,
 <th>Gender</th>,
 <th>City</th>,
 <th>State</th>,
 <th>Time</th>,
 <th>Gun Time</th>]

In [94]:
# Extraímos texto entre tags html dos cabeçalhos de tabela (extraímos o nome)
all_header = []
col_str = str(col_labels)
cleantext2 = bs(col_str, "lxml").get_text()
all_header.append(cleantext2)

In [95]:
print(all_header)

['[Place, Bib, Name, Gender, City, State, Time, Gun Time]']


In [96]:
# Convertendo a lista de cabeçalhos em um dataframe do pandas
df2 = pd.DataFrame(all_header)

In [97]:
df2.head()

,0
0,"[Place, Bib, Name, Gender, City, State, Time, ..."


In [98]:
# Dividindo a coluna "0" em várias colunas na posição de vírgula para todas as linhas
df3 = df2[0].str.split(',', expand = True)

In [99]:
df3.head()

,0,1,2,3,4,5,6,7
0,[Place,Bib,Name,Gender,City,State,Time,Gun Time]


In [100]:
# Concatenando os 2 dataframes
frames = [df3, df1]
df4 = pd.concat(frames)

In [101]:
df4.head(10)

,0,1,2,3,4,5,6,7
0,[Place,Bib,Name,Gender,City,State,Time,Gun Time]
0,],None,None,None,None,None,None,None
1,Finishers:,1458],None,None,None,None,None,None
2,Male:,771],None,None,None,None,None,None
3,Female:,687],None,None,None,None,None,None
4,],None,None,None,None,None,None,None
5,1,2320,\r\n\r\n DANIEL M HINCKLEY...,M,HILLSBORO,OR,16:42,16:44]
6,2,2335,\r\n\r\n KORY F GRAY\r\n\r...,M,HILLSBORO,OR,17:34,17:35]
7,3,1770,\r\n\r\n FILIP SCHMOLE\r\n...,M,PORTLAND,OR,18:13,18:14]
8,4,2584,\r\n\r\n TRENTON C ROLLING...,M,PORTLAND,OR,18:32,18:35]


In [102]:
# Atribuindo a primeira coluna como cabeçalho
df5 = df4.rename(columns = df4.iloc[0])

In [103]:
df5.head()

,[Place,Bib,Name,Gender,City,State,Time,Gun Time]
0,[Place,Bib,Name,Gender,City,State,Time,Gun Time]
0,],None,None,None,None,None,None,None
1,Finishers:,1458],None,None,None,None,None,None
2,Male:,771],None,None,None,None,None,None
3,Female:,687],None,None,None,None,None,None


In [104]:
# Shape
df5.shape

(1464, 8)

In [105]:
# Obtendo uma visão geral dos dados para análise
df5.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1464 entries, 0 to 1462
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   [Place      1464 non-null   object
 1    Bib        1462 non-null   object
 2    Name       1459 non-null   object
 3    Gender     1459 non-null   object
 4    City       1459 non-null   object
 5    State      1459 non-null   object
 6    Time       1459 non-null   object
 7    Gun Time]  1459 non-null   object
dtypes: object(8)
memory usage: 102.9+ KB


In [106]:
# Verificando valores ausentes
df5.isnull().sum()

[Place        0
 Bib          2
 Name         5
 Gender       5
 City         5
 State        5
 Time         5
 Gun Time]    5
dtype: int64

In [107]:
# Removendo todas as linhas com valores ausentes
df6 = df5.dropna(axis = 0, how = 'any')

In [108]:
# Descartando o cabeçalho da tabela replicada como a primeira linha no df5
df7 = df6.drop(df6.index[0])

In [109]:
# Renomeando as colunas [Place e Team] 
df7.rename(columns = {'[Place': 'Place'}, inplace = True)
df7.rename(columns = {' Team]': 'Team'}, inplace = True)

In [110]:
# Limpeza final dos dados - remoção do colchete de fechamento das células na coluna "Team".
df7['Team'] = df7['Team'].str.strip(']')

KeyError: 'Team'

In [ ]:
df7.head()

## Análise de Dados

In [ ]:
# Convertendo a coluna "Chip Time" em apenas minutos
# Chip time é o tempo total de corrida medido com a leitura do sensor RFID no selo da camisa do participante
time_list = df7[' Chip Time'].tolist()

In [ ]:
# Visualizando uma amostra de dados
time_list[1:5]

In [ ]:
# Lista para receber o resultado da conversão
time_mins = []

In [ ]:
# Loop para conversão
for i in time_list:
    i = i.strip(' ')
    if len(i)!=7:
        i = '0:' + i
    h, m, s = i.split(':')
    math = (int(h) * 3600 + int(m) * 60 + int(s))/60
    time_mins.append(math)

Convertendo a lista novamente em um dataframe e criamos uma nova coluna ("Runner_mins") como tempo total de corrida, mas agora em minutos.

In [ ]:
# Nova coluna
df7['Runner_mins'] = time_mins

In [ ]:
df7.head()

In [ ]:
df7.dtypes

In [ ]:
# Cálculo de estatísticas para colunas numéricas no dataframe
df7.describe(include = [np.number])

**Análise**:

- O tempo médio de chip para todos os corredores foi de ~ 60 minutos (a média na tabela acima).
- O corredor 10K mais rápido terminou em 36,35 minutos (valor mínimo na tabela acima).
- O corredor mais lento terminou em 101,30 minutos (valor máximo na tabela acima).

Vamos criar um boxplot para sumarizar nossa análise.

In [ ]:
# Plot
ax = sns.boxplot(x = df7["Runner_mins"], palette = "Set3", orient = "v")
plt.ylabel('Chip Time')
plt.xlabel('Runners')
plt.show()

Acima estão as estatísticas de resumo de dados para os corredores mostrados em um boxplot: Máximo, mínimo, médio, primeiro quartil, terceiro quartil, incluindo valores extremos.

Criaremos agora um gráfico de distribuição dos tempos dos chips dos corredores usando a biblioteca seaborn.

In [ ]:
# Plot
x = df7['Runner_mins']
ax = sns.distplot(x, hist = True, kde = True, rug = False, color = 'blue', bins = 25, hist_kws = {'edgecolor':'black'})
plt.show()

A distribuição da variável parece quase normal. Mas vamos realizar um teste estatístico para confirmar.

Um ponto de decisão importante ao trabalhar com uma amostra de dados é usar métodos estatísticos paramétricos ou não paramétricos.

Os métodos estatísticos paramétricos assumem que os dados têm uma distribuição conhecida e específica, geralmente uma distribuição gaussiana (normal). Se uma amostra de dados não é gaussiana, as suposições dos testes estatísticos paramétricos são violadas e métodos estatísticos não paramétricos devem ser utilizados.

Há várias técnicas que você pode usar para verificar se sua amostra de dados se desvia de uma distribuição gaussiana, chamada testes de normalidade.

O pacote stats do SciPy fornece uma função para um dos testes de normalidade mais comuns, o Shapiro-Wilk Test. Usaremos ainda a função normality do pacote Pengouin.

O Shapiro-Wilk é um teste de hipóteses, que testa essas duas hipóteses:

**A hipótese nula deste teste é que a população é normalmente distribuída. Assim, se o valor-p for menor que o nível alfa escolhido (normalmente definido em 0,05), a hipótese nula será rejeitada e há evidências de que os dados testados não são normalmente distribuídos.**

O resultado do teste de Shapiro-Wilk deve ser interpretado com cautela no caso de amostras grandes. 

In [ ]:
# Valor de x
x = df7['Runner_mins']

In [ ]:
# Teste de normalidade com Scipy
stats.shapiro(x)

In [ ]:
# Teste de normalidade com Pingouin
pg.normality(x)

**Análise**:

- A variável não segue uma distribuição normal, uma vez que o valor-p é menor que 0.05 e assim há evidências estatísticas para rejeitar a hipótese nula. A função normality() do pacote Pingouin coloca essa informação na última coluna.

Vamos agora descobrir se houve alguma diferença de desempenho entre homens e mulheres de várias faixas etárias.

In [ ]:
# Separando os dados
f_runners = df7.loc[df7[' Gender'] == ' F']['Runner_mins']
m_runners = df7.loc[df7[' Gender'] == ' M']['Runner_mins']

In [ ]:
# Criaremos os 2 plots em uma só figura para facilitar a comparação
sns.distplot(f_runners, hist = True, kde = True, rug = False, hist_kws = {'edgecolor':'black'}, label = 'Mulher')
sns.distplot(m_runners, hist = False, kde = True, rug = False, hist_kws = {'edgecolor':'black'}, label = 'Homem')
plt.legend()

**Análise**:

- A distribuição indica que as mulheres eram mais lentas que os homens, em média.

In [ ]:
# Computando estatísticas resumidas para homens e mulheres separadamente usando o método groupby ()
g_stats = df7.groupby(" Gender", as_index = True).describe()
g_stats

**Análise**:
    
- O tempo médio de chip para todas as mulheres e homens foi de ~ 66 minutos e ~ 58 minutos, respectivamente, comprovando a análise do gráfico. 

Um boxplot ajuda a comparar as amostras de homens e mulheres.

In [ ]:
# Plot
df7.boxplot(column = 'Runner_mins', by = ' Gender')
plt.ylabel('Chip Time')
plt.suptitle("")

Nossa fonte de dados possui resultados de diversas corridas. Material em abundância para você praticar seus conhecimentos em análise de dados.

# Fim